# Article 3: RAG Evaluation - Benchmark Analysis

Reads `results/data/article_03_benchmarks.json` (multi-evaluator schema produced by
`benchmarks/run_article_03.py`) and renders charts for whichever evaluators actually
ran. The runner skips evaluators whose API keys are missing and reports only what
ran; this notebook follows the same rule and never fabricates numbers.

Hard-fails with `FileNotFoundError` if the JSON is missing. Generate the data first:

```bash
uv run python benchmarks/run_article_03.py --runs 3
```

Charts produced:
1. `01_judge_metric_means.png` - Per-evaluator metric means with std error bars
2. `02_metric_stability.png` - Per-run means scatter (cross-run stability)
3. `03_pipeline_latency.png` - Retrieval+generation latency (min/mean/max + std)


In [1]:
"""Article 3: RAG evaluation analysis notebook.

Reads the multi-evaluator benchmark JSON and renders charts for the evaluators
that actually ran. Hard-fails on missing input rather than fabricating data.
"""
from __future__ import annotations

import json
from pathlib import Path

import matplotlib
# Agg backend keeps the notebook headless-safe under `jupyter nbconvert --execute`
# in CI: no display server required, charts go straight to PNG.
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path("..").resolve()
DATA = ROOT / "results" / "data"
CHARTS = ROOT / "results" / "charts" / "article_03"
CHARTS.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 150,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.4,
    "font.size": 11,
})

# Distinct hues per evaluator so multi-evaluator runs read clearly.
EVAL_COLORS = {
    "ragas": "#3498db",
    "llm_judge": "#e67e22",
}
EVAL_DISPLAY = {
    "ragas": "RAGAS",
    "llm_judge": "LLM-as-Judge",
}

print("Setup complete. Charts will be written to:", CHARTS)


Setup complete. Charts will be written to: /Users/jagdish/work/intellij/py-ws/agentic-ai-stress-suite/results/charts/article_03


## Load Benchmark JSON

The runner emits a multi-evaluator schema. Each entry in `results[]` carries
wall-time, sample count, and per-metric mean/std/per_run_means across `runs` runs.
We hard-fail if the file is missing - notebooks should never synthesise numbers
that look real.


In [2]:
bench_json = DATA / "article_03_benchmarks.json"

if not bench_json.exists():
    raise FileNotFoundError(
        f"Required benchmark output missing: {bench_json}. "
        "Run `uv run python benchmarks/run_article_03.py` first."
    )

data = json.loads(bench_json.read_text())
results = data["results"]
settings = data["settings"]
pipeline = data["pipeline"]
dataset = data["dataset"]

if not results:
    raise RuntimeError(
        "No evaluator results in benchmark JSON. "
        "Likely cause: every evaluator was skipped (missing API keys). "
        "Re-run with at least one evaluator enabled."
    )

print(f"Benchmark:    {data['benchmark']}")
print(f"Pipeline:     {pipeline['name']} ({pipeline['embedding_model']})")
print(f"Evaluators:   {len(results)} ({', '.join(r['name'] for r in results)})")
print(f"Queries:      {dataset['queries_evaluated']}/{dataset['num_queries']}  (errors: {dataset['pipeline_errors']})")
print(f"Runs/eval:    {settings['runs']}")
print(f"Top-K:        {settings['top_k']}")


Benchmark:    article_03_evaluation
Pipeline:     naive_rag (BAAI/bge-base-en-v1.5)
Evaluators:   2 (ragas, llm_judge)
Queries:      30/30  (errors: 0)
Runs/eval:    3
Top-K:        5


## Build DataFrame

One row per (evaluator, metric) with mean/std/per_run_means. Cleaner to slice
for plots and easy to drop into the article body as a summary table.


In [3]:
rows = []
for evaluator in results:
    name = evaluator["name"]
    for metric_name, metric_data in evaluator["metrics"].items():
        rows.append({
            "evaluator": name,
            "metric": metric_name,
            "mean": metric_data["mean"],
            "std": metric_data["std"],
            "n_runs": metric_data["n_runs"],
            "n_samples": metric_data["n_samples"],
            "per_run_means": metric_data["per_run_means"],
        })

df = pd.DataFrame(rows)
print(df.drop(columns=["per_run_means"]).to_string(index=False))


evaluator             metric     mean      std  n_runs  n_samples
    ragas answer_correctness 0.436292 0.021846       3         30
    ragas       faithfulness 0.802932 0.001378       3         30
    ragas   answer_relevancy 0.937042 0.000288       3         30
    ragas  context_precision 0.740741 0.020580       3         30
llm_judge        correctness 0.984444 0.003143       3         30
llm_judge       completeness 0.966667 0.005443       3         30
llm_judge       groundedness 0.982222 0.006285       3         30
llm_judge          relevance 0.997778 0.003143       3         30


## Chart 1: Evaluator metric means

Side-by-side panels, one per evaluator, since RAGAS and LLM-judge use
disjoint metric names (RAGAS: `answer_correctness`, `faithfulness`,
`answer_relevancy`, `context_precision`; Judge: `correctness`,
`completeness`, `groundedness`, `relevance`). Both panels share the 0-1
y-axis so cross-evaluator disagreement reads at a glance: where one
evaluator says 0.4 and the other says 0.98 on the same RAG output, the
gap is the headline of this article.


In [4]:
evaluators_in_order = [r["name"] for r in results]
n_evals = len(evaluators_in_order)

fig, axes = plt.subplots(
    1, n_evals,
    figsize=(6 * n_evals, 5.5),
    sharey=True,
    squeeze=False,
)

for ax, evaluator in zip(axes[0], evaluators_in_order):
    sub = df[df["evaluator"] == evaluator].sort_values("metric")
    metrics = sub["metric"].tolist()
    means = sub["mean"].tolist()
    stds = sub["std"].tolist()
    color = EVAL_COLORS.get(evaluator, "#7f8c8d")
    x = np.arange(len(metrics))
    bars = ax.bar(
        x, means, 0.6,
        yerr=stds, capsize=5,
        color=color, edgecolor="black", linewidth=0.5,
    )
    for bar, m in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width() / 2, m + 0.02, f"{m:.3f}",
                ha="center", va="bottom", fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels(metrics, rotation=15, ha="right")
    ax.set_title(EVAL_DISPLAY.get(evaluator, evaluator))
    ax.set_ylim(0, 1.05)

axes[0][0].set_ylabel("Score (0-1)")
fig.suptitle(
    f"Evaluator metric means (mean +/- std across {settings['runs']} runs, "
    f"n={dataset['queries_evaluated']} queries)"
)

plt.tight_layout()
out = CHARTS / "01_judge_metric_means.png"
plt.savefig(out, dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {out}")


Saved: /Users/jagdish/work/intellij/py-ws/agentic-ai-stress-suite/results/charts/article_03/01_judge_metric_means.png


## Chart 2: Cross-run stability

Same per-evaluator panel layout as chart 1. Each marker is one run's mean
for that metric; the short horizontal bar is the cross-run mean. Tightly
clustered markers = stable evaluator; spread = high run-to-run variance
(judge nondeterminism, sampling effects). At temperature=0 the judge should
be near-deterministic; RAGAS uses non-zero temperature for several metrics
so some variance is expected there.


In [5]:
fig, axes = plt.subplots(
    1, n_evals,
    figsize=(6 * n_evals, 5.5),
    sharey=True,
    squeeze=False,
)

for ax, evaluator in zip(axes[0], evaluators_in_order):
    sub = df[df["evaluator"] == evaluator].sort_values("metric")
    metrics = sub["metric"].tolist()
    color = EVAL_COLORS.get(evaluator, "#7f8c8d")
    for metric_idx, (_, row) in enumerate(sub.iterrows()):
        per_run = row["per_run_means"]
        ax.scatter(
            [metric_idx] * len(per_run), per_run,
            color=color, s=70, alpha=0.75, edgecolor="black", linewidth=0.5,
        )
        ax.hlines(row["mean"], metric_idx - 0.18, metric_idx + 0.18,
                  color=color, linewidth=2.5)
    ax.set_xticks(range(len(metrics)))
    ax.set_xticklabels(metrics, rotation=15, ha="right")
    ax.set_title(EVAL_DISPLAY.get(evaluator, evaluator))
    ax.set_ylim(0, 1.05)

axes[0][0].set_ylabel("Per-run mean score")
fig.suptitle(
    f"Cross-run stability ({settings['runs']} runs; markers = run means, bar = aggregate)"
)

plt.tight_layout()
out = CHARTS / "02_metric_stability.png"
plt.savefig(out, dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {out}")


Saved: /Users/jagdish/work/intellij/py-ws/agentic-ai-stress-suite/results/charts/article_03/02_metric_stability.png


## Chart 3: Pipeline latency

Retrieval + generation latency for the underlying pipeline (separate from the
evaluator wall-time, which is reported as a footnote). The std bracket on the
mean bar shows query-to-query spread.


In [6]:
lat = pipeline["latency_ms"]

fig, ax = plt.subplots(figsize=(8, 5))
labels = ["min", "mean", "max"]
values = [lat["min"], lat["mean"], lat["max"]]
bars = ax.bar(labels, values, color=["#27ae60", "#3498db", "#e74c3c"],
              edgecolor="black", linewidth=0.5)

mean_idx = labels.index("mean")
ax.errorbar(mean_idx, lat["mean"], yerr=lat["std"], fmt="none",
            ecolor="black", capsize=8, linewidth=1.5)

for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, v + max(values) * 0.01,
            f"{v:.0f} ms", ha="center", va="bottom", fontsize=10)

ax.set_ylabel("Latency (ms)")
ax.set_title(f"Pipeline latency: {pipeline['name']} (n={dataset['queries_evaluated']} queries)")

wall_lines = [
    f"{EVAL_DISPLAY.get(r['name'], r['name'])}: {r['wall_time_seconds']:.1f}s wall ({r['num_runs']} runs)"
    for r in results
]
ax.text(0.02, 0.98, "\n".join(wall_lines), transform=ax.transAxes,
        ha="left", va="top", fontsize=9,
        bbox=dict(boxstyle="round,pad=0.4", facecolor="#f8f9fa", edgecolor="#dee2e6"))

plt.tight_layout()
out = CHARTS / "03_pipeline_latency.png"
plt.savefig(out, dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {out}")


Saved: /Users/jagdish/work/intellij/py-ws/agentic-ai-stress-suite/results/charts/article_03/03_pipeline_latency.png


## Export Summary JSON

Mirrors the benchmark JSON shape but with explicit Python casts so downstream
consumers (article body, CI gates) get clean floats/ints - no numpy scalars.


In [7]:
summary = {
    "benchmark": data["benchmark"],
    "pipeline": {
        "name": pipeline["name"],
        "embedding_model": pipeline["embedding_model"],
        "latency_ms": {k: float(v) for k, v in lat.items()},
    },
    "dataset": {
        "num_queries": int(dataset["num_queries"]),
        "queries_evaluated": int(dataset["queries_evaluated"]),
        "pipeline_errors": int(dataset["pipeline_errors"]),
    },
    "settings": {
        "top_k": int(settings["top_k"]),
        "runs": int(settings["runs"]),
        "judge_temperature": float(settings["judge_temperature"]),
        "ragas_llm": settings.get("ragas_llm"),
    },
    "evaluators": {},
}
for evaluator in results:
    summary["evaluators"][evaluator["name"]] = {
        "wall_time_seconds": float(evaluator["wall_time_seconds"]),
        "num_runs": int(evaluator["num_runs"]),
        "samples_scored": int(evaluator["samples_scored"]),
        "metrics": {
            metric_name: {
                "mean": float(metric_data["mean"]),
                "std": float(metric_data["std"]),
                "n_runs": int(metric_data["n_runs"]),
                "n_samples": int(metric_data["n_samples"]),
                "per_run_means": [float(v) for v in metric_data["per_run_means"]],
            }
            for metric_name, metric_data in evaluator["metrics"].items()
        },
    }

summary_path = DATA / "article_03_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))
print(f"Wrote summary: {summary_path}")


Wrote summary: /Users/jagdish/work/intellij/py-ws/agentic-ai-stress-suite/results/data/article_03_summary.json


## Notes

- **Evaluators present in this run**: see the `evaluators_in_order` list above.
  The runner skips evaluators whose API keys are missing/placeholder rather than
  fabricating numbers; the article body must narrate honestly which ran.
- **No drift detection / golden-set A-B charts here**: those move to Article 6 / 7
  where the supporting infrastructure exists. Earlier drafts of this notebook
  generated synthetic versions of those charts - they were removed.
- **Score scale**: All metrics are 0-1. If the baseline `naive_rag` pipeline
  scores low (e.g. groundedness near 0), report it as-is - that is the empirical
  finding the article exists to surface, and Article 2's advanced retrieval is
  the comparison that gives it shape.
